# ResNet(Residual Network)

## 이론

### 개요

기존 딥러닝에서 인공 신경망은 네트워크가 많아질 수록 성능이 좋아질 것이라 생각했지만, 두 가지 문제가 발생함.

- 성능 저하(Degradation Problem)
    - 레이어를 하나 더 만들었는데 오히려 학습/테스트 오차 증가
    - 학습 데이터에 과적합되는게 아니라, 그냥 최적화 자체가 어려워짐
- Vanishing Gradient
    - 레이어가 중첩되며 역전파가 중첩될 때 마다 Gradient가 점점 줄어듬
    - 초기 레이어는 학습이 거의 되지 않음

두 가지 문제가 발생해, 차이를 이용해 학습(Residual Learning)이 추가됨

일반적인 신경망은 `H(x)`로 학습을 하는데, 

ResNet은 `H(x) = F(x) + x; F(x) = H(x) - x`으로 계산해, 완전한 함수 H(X)를 학습하는게 아니라, 입력값과의 차이만을 이용해 학습을 진행하게 수정했다. 

### Residual Block

아래의 ResNet의 블록 구조를 Skip Connection이라고 부른다. (y=F(x,Wi​)+x)
- F(x, Wi) : Convolution Layer들의 출력
- x = 입력
- y = 최종 출력

```
x ───► [Conv → BN → ReLU → Conv → BN] ───► + ───► ReLU
   └──────────────────────────────────────┘
```

### 이렇게 했을 때 장점

`y = H(x)`으로 했을 때 `dL/dx`를 할 때 소실되는 gradient(기울기)가 사라지지 않고 그대로 전달될 수 있었다. 
불필요한 학습이라면, `F(x) = 0`으로 학습시켜 x 그대로 전달할 수 있게 된다. (`y = 0 + x`)

레이어에 따라 18, 34, 50, 101, 152가 존재한다.(50이상으로는 BottleNeck을 이용함)

### BottleNeck

## 실습 

![res_net_1.png](../pytorch_study/images/res_net_1.png)

- 아키텍처 중 좌측 상단의 구조가 가장 성능이 좋게 나왔다. 
- resnet(18, 34, 50, 101, 152)를 만들 수 있게 되어 있음
- 3 * 244 * 244 입력을 기준으로 되어있음
- input size가 다르면 ResNet에 어떻게 적용할지? -> 

In [2]:
import torch

In [3]:
import torch.nn as nn
import torch.utils.model_zoo as model_zoo

__all__ = ['ResNet', 'resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152'] # 성능 테스트를 위한 resnet들
model_urls = {
    'resnet18' : 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
    'resnet34' : 'https://download.pytorch.org/models/resnet34-333f7ec4.pth',
    'resnet50' : 'https://download.pytorch.org/models/resnet50-19c8e357.pth',
    'resnet101' : 'https://download.pytorch.org/models/resnet101-5d3b4d8f.pth',
    'resnet152' : 'https://download.pytorch.org/models/resnet152-b121ed2d.pth',
}

In [4]:
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=1, bias=False)


![res_net_2.png](../pytorch_study/images/res_net_2.png)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride
    
    def forward(self, x):
        identity = x
        
        out = self.conv1(x) # 3x3 stride = stride
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out) # 3x3 stride = 1
        out = self.bn2(out)
        # stirde가 1이 아닌 값이 들어왔을 때 x.shape이 3 * 64 * 64라면, 
        # self.conv1(x) = 3x3 stride 2, 
        # conv2 = 3x3 stride = 1이 된다.
        # 이 때 덧셈이 불가능한 상태가 되어 downsample의 조건이 추가된다. 
        if self.downsample is not None: 
            identity = self.downsample(x)
            
        out += identity
        out = self.relu(out)
        
        return out

### `self.downsample` 조건이 필요한 이유

Skip Connection의 핵심은 `out += identity`, 즉 **Conv 레이어를 통과한 출력(out)과 원래 입력(identity)을 더하는 것**이다.

덧셈이 가능하려면 두 텐서의 **shape이 완전히 일치**해야 한다.

---

#### shape이 불일치하는 두 가지 경우

**1. stride > 1 → 공간 크기(H, W) 불일치**

| | shape |
|---|---|
| 입력 x | `(B, C, 64, 64)` |
| conv1 (stride=2) 통과 후 | `(B, C, 32, 32)` |
| conv2 통과 후 out | `(B, C, 32, 32)` |
| identity (원본 x) | `(B, C, 64, 64)` ← 크기가 다름! |

`out (32x32) + identity (64x64)` → **shape 불일치로 덧셈 불가**

**2. 채널 수 변화 → 채널(C) 불일치**

레이어가 넘어갈 때 채널이 `64 → 128`처럼 증가하는 경우,  
out의 채널은 128인데 identity의 채널은 64이므로 마찬가지로 **덧셈 불가**

---

#### `downsample`의 역할

`downsample`은 identity(원본 x)의 shape을 out에 맞게 변환해주는 레이어로, 일반적으로 `conv1x1 + BatchNorm`으로 구성된다.

```python
downsample = nn.Sequential(
    conv1x1(inplanes, planes * block.expansion, stride),  # 채널 수 맞춤 + stride로 공간 크기 맞춤
    nn.BatchNorm2d(planes * block.expansion),
)
```

- `stride`로 공간 크기(H, W)를 out과 동일하게 맞춤
- `planes * expansion`으로 채널 수를 out과 동일하게 맞춤

---

#### 정리

```
stride=1, 채널 변화 없음  →  shape 일치  →  downsample 불필요 (None)
stride>1 또는 채널 변화   →  shape 불일치 →  downsample로 identity shape 변환 후 덧셈
```

결국 `if self.downsample is not None` 조건은,  
**shape이 맞지 않는 상황에서만 identity를 변환**하여 Skip Connection이 항상 올바르게 동작하도록 보장하는 장치다.

In [ ]:
class BottleNeck(nn.Module):
    expansion = 4
    
    def __init__(self, inplanes, planes, stride = 1, downsample = None):
        super(BottleNeck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        
        out = self.conv3(out)
        out = self.bn3(out)
        
        if self.downsample is not None: 
            identity = self.downsample(x)
            
        out += identity
        out = self.relu(out)
        
        return out